In [24]:
from datetime import date
import polars as pl
from data_engineering.adsb.read_adsb import read_adsb
from data_engineering.adsbx.get_flights import get_adsbx_flights_for_day
from data_engineering.bts.read_bts_ontime_to_flights import get_bts_flights_for_day
from data_engineering.flights.sfdps_to_flights import get_sfdps_flights_day
from flights.flights_match_adsb import flights_match_adsb, get_matching_icaos_in_flights
test_date = date(2026,3,1)
df_sfdps = get_sfdps_flights_day(test_date)
df_adsbx = get_adsbx_flights_for_day(test_date)
# df_model = pl.read_parquet('/Volumes/T2-SSD/planequery/data/flights/trad-model/2026-03-01.parquet')
# df_model = pl.read_parquet('/Volumes/T2-SSD/planequery/data/flights/trad-model/v2/flights-adsbx.parquet')
df_model = pl.read_parquet('/Volumes/T2-SSD/planequery/data/flights/trad-model/v2/flights-adsblol.parquet')
expr = pl.col("takeoff_time").dt.date() == test_date 
df_sfdps = df_sfdps.filter(expr)
# df_adsbx = df_adsbx.filter(expr)
df_model = df_model.filter(expr)
df_gold = df_sfdps
# df_gold = get_matching_icaos_in_flights(df_sfdps, read_adsb(test_date))
df_model = df_model.filter(pl.col("icao").is_in(df_gold.get_column("icao").unique().implode()))

In [27]:
df_model.filter(pl.col("takeoff_airport_ident_distance_km") > 30).filter(pl.col("ladd"))

icao,callsign,registration,takeoff_time,takeoff_airport_ident,takeoff_airport_ident_distance_km,landing_time,landing_airport_ident,landing_airport_ident_distance_km,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,f64,datetime[ms],str,f64,bool,bool,bool,bool,str,str,str,str
"""0d0cf3""","""""","""""",2026-03-01 19:56:32.820,"""MYBC""",46.987173,2026-03-01 21:03:28.830,"""KFLL""",0.141647,false,true,false,false,"""""","""""","""""",""""""
"""a0012f""","""""","""""",2026-03-01 15:51:57.910,"""TUPA""",196.844984,2026-03-01 19:45:49.800,"""KPSM""",1.789154,false,true,false,false,"""""","""""","""""",""""""
"""a003a8""","""""","""""",2026-03-01 19:44:39.670,"""MYBC""",66.133388,2026-03-01 20:52:15.660,"""KFXE""",1.221067,false,true,false,false,"""""","""""","""""",""""""
"""a05277""","""""","""""",2026-03-01 20:06:41.220,"""MYBC""",35.550002,2026-03-01 21:05:21.870,"""KOPF""",1.374678,false,true,false,false,"""""","""""","""""",""""""
"""a05483""","""""","""""",2026-03-01 20:44:29.665,"""TNCM""",70.58052,2026-03-01 23:53:00.380,"""KBCT""",0.292478,false,true,false,false,"""""","""""","""""",""""""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ace478""","""""","""""",2026-03-01 14:37:13.575,"""MYAT""",32.160152,2026-03-01 17:08:31.500,"""KJYO""",0.654658,false,true,false,false,"""""","""""","""""",""""""
"""ad8c37""","""""","""""",2026-03-01 19:29:41.085,"""TIST""",35.717463,2026-03-01 22:31:34.100,"""KPBI""",0.823004,false,true,false,false,"""""","""""","""""",""""""
"""ada2a5""","""""","""""",2026-03-01 18:59:45.940,"""US-1194""",231.492759,2026-03-01 20:00:49.180,"""KMSY""",1.494303,false,true,false,false,"""""","""""","""""",""""""


In [18]:
from flights.flights_comparison import df_flights_comparision, df_flights_comparison_stats, flights_comparison
from datetime import datetime, time
start_dt = datetime.combine(test_date, time.min)
end_dt = datetime.combine(test_date, time.max)
df = df_flights_comparision(df_model, df_gold, start_dt=start_dt, end_dt=end_dt, compare_airports=False)
stats = df_flights_comparison_stats(df)
print(stats)

{'true_positive': 7571, 'false_positive': 319, 'false_negative': 177, 'precision': 0.9595690747782003, 'recall': 0.9771553949406299, 'f1': 0.9682823890523086, 'airport_ident_match_pct': 0.9228635583146216}


In [19]:
COLUMNS = [
    "icao",
    "takeoff_time",
    "takeoff_airport_ident",
    "takeoff_airport_ident_distance_km",
    "landing_time",
    "landing_airport_ident",
    "landing_airport_ident_distance_km",
    "takeoff_time_df1",
    "takeoff_airport_ident_df1",
    "landing_time_df1",
    "landing_airport_ident_df1",
    "same_takeoff_airport_ident",
    "same_landing_airport_ident",
    "same_airport_ident",
    "match_status",
]
df = df.select(COLUMNS)

In [20]:
print(len(df.filter(pl.col("match_status") == "both").filter(pl.col("same_takeoff_airport_ident") == False)))
print(len(df.filter(pl.col("match_status") == "both").filter(pl.col("same_landing_airport_ident") == False)))

207
436


In [12]:
print(len(df))
print(len(df.filter(pl.col("ladd"))))
print(len(df.filter(pl.col("match_status") == "both").filter(pl.col("same_airport_ident") == False)))
print(len(df.filter(pl.col("match_status") == "both").filter(pl.col("ladd")).filter(pl.col("same_airport_ident") == False)))

35699
2566
2590
256


In [23]:
df.filter(pl.col("match_status") == "both").filter(pl.col("same_landing_airport_ident") == False).filter(pl.col("landing_airport_ident_distance_km") > 20)

icao,takeoff_time,takeoff_airport_ident,takeoff_airport_ident_distance_km,landing_time,landing_airport_ident,landing_airport_ident_distance_km,takeoff_time_df1,takeoff_airport_ident_df1,landing_time_df1,landing_airport_ident_df1,same_takeoff_airport_ident,same_landing_airport_ident,same_airport_ident,match_status
str,datetime[ms],str,f64,datetime[ms],str,f64,datetime[ms],str,datetime[ms],str,bool,bool,bool,str
"""a099fd""",2026-03-01 15:07:12.011,"""KFXE""",0.23145,2026-03-01 18:37:08.040,"""TNCM""",37.836662,2026-03-01 15:07:00,"""KFXE""",2026-03-01 18:55:00,"""TQPF""",true,false,false,"""both"""
"""a19634""",2026-03-01 14:23:27.064,"""KXBP""",1.565295,2026-03-01 16:22:11.260,"""KELD""",30.703875,2026-03-01 14:27:00,"""KXBP""",2026-03-01 16:50:00,"""KBQP""",true,false,false,"""both"""
"""a1df50""",2026-03-01 14:41:12.951,"""KPMP""",0.18504,2026-03-01 18:32:55.349,"""TNCM""",37.668779,2026-03-01 14:41:00,"""KPMP""",2026-03-01 18:53:00,"""TQPF""",true,false,false,"""both"""
"""a37e7b""",2026-03-01 16:44:53.205,"""KAPA""",0.268134,2026-03-01 19:23:21.931,"""KBQP""",20.329767,2026-03-01 16:52:00,"""KAPA""",2026-03-01 19:36:00,"""KMLU""",true,false,false,"""both"""
"""a5dbe3""",2026-03-01 13:41:28.747,"""KEDC""",1.207675,2026-03-01 14:48:22.625,"""3XS4""",23.004323,2026-03-01 13:41:00,"""KEDC""",2026-03-01 15:09:00,"""3R7""",true,false,false,"""both"""
"""ac67fc""",2026-03-01 18:43:41.518,"""KMIA""",0.615629,2026-03-01 19:46:49.060,"""MYMM""",58.43478,2026-03-01 18:44:00,"""KMIA""",2026-03-01 20:07:00,"""MBPV""",true,false,false,"""both"""
"""c06135""",2026-03-01 13:26:58.049,"""KFXE""",0.51085,2026-03-01 14:28:25.945,"""MYNN""",39.231757,2026-03-01 13:27:00,"""KFXE""",2026-03-01 15:16:00,"""MYEF""",true,false,false,"""both"""


In [13]:
df2 = df.filter(pl.col("match_status") == "both").filter(pl.col("ladd")).filter(pl.col("same_airport_ident") == False)
df2[20:60]

icao,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,takeoff_time_df1,takeoff_airport_ident_df1,landing_time_df1,landing_airport_ident_df1,same_takeoff_airport_ident,same_landing_airport_ident,same_airport_ident,match_status,ladd,pia
str,datetime[ms],str,datetime[ms],str,datetime[ms],str,datetime[ms],str,bool,bool,bool,str,bool,bool
"""a0fd7b""",2026-03-01 20:22:58.860,"""KFTW""",2026-03-01 22:30:28.406,"""US-6634""",2026-03-01 20:23:00,"""KFTW""",2026-03-01 21:48:00,"""KTDW""",true,false,false,"""both""",true,false
"""a10f22""",2026-03-01 01:38:28.888,"""KMCE""",2026-03-01 02:43:06.181,"""KLVK""",2026-03-01 01:39:00,"""KMCE""",2026-03-01 02:41:00,"""KMOD""",true,false,false,"""both""",true,false
"""a10f22""",2026-03-01 18:44:43.060,"""KSAC""",2026-03-01 19:50:29.001,"""KLVK""",2026-03-01 19:10:00,"""KVCB""",2026-03-01 19:50:00,"""LVK""",false,false,false,"""both""",true,false
"""a10f22""",2026-03-01 21:15:41.613,"""KLVK""",2026-03-01 23:09:58.748,"""KLVK""",2026-03-01 21:17:00,"""KLVK""",2026-03-01 22:17:00,"""DWA""",true,false,false,"""both""",true,false
"""a10fff""",2026-03-01 23:03:13.728,"""KTVC""",2026-03-01 23:10:13.837,"""KTVC""",2026-03-01 23:20:00,"""KTVC""",2026-03-01 23:58:00,"""KYIP""",true,false,false,"""both""",true,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""a259bc""",2026-03-01 16:50:43.053,"""KVGT""",2026-03-01 19:26:58.942,"""KVGT""",2026-03-01 16:50:00,"""MMM184013""",2026-03-01 18:48:00,"""KBVU""",false,false,false,"""both""",true,false
"""a25df2""",2026-03-01 20:14:28.926,"""KOLV""",2026-03-01 21:59:47.534,"""KOLV""",2026-03-01 20:14:00,"""KOLV""",2026-03-01 21:18:00,"""M04""",true,false,false,"""both""",true,false
"""a25e9b""",2026-03-01 16:29:13.748,"""KCXO""",2026-03-01 19:38:27.059,"""KBTR""",2026-03-01 17:17:00,"""KBPT""",2026-03-01 19:17:00,"""HDC""",false,false,false,"""both""",true,false


In [47]:
df_adsbx.filter(pl.col("icao") == "a00581")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool
"""a00581""",null,"""N100JE""",2026-03-01 15:55:30,"""KX51""",2026-03-01 17:57:00,"""KVDF""",false,false,false,false


# What am I trying to do? 
# My model and adsbx could be missing exact same flights so there's this problem here. 
# This may only be good for testing adsb matching algoirhtm. 
# Ok with BTS see if I do intersection of df_model and df_adsbx if all the flights not in bts are because bts is wrong